# EDA | Mundial de Fútbol 2026

Análisis exploratorio de rendimiento ofensivo, posesión, disciplina y formaciones tácticas. Este notebook reúne los gráficos nuevos trabajados en Colab.
Los análisis que dependen del resultado utilizan partidos con marcador disponible. Cada partido se representa desde la perspectiva de ambos equipos, local y visitante.

## 1. Importación y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# En Google Colab, descomenta para cargar el CSV:
# from google.colab import files
# archivos = files.upload()
# nombre_archivo = next(iter(archivos))
# df = pd.read_csv(nombre_archivo)

# Alternativa si ejecutas el notebook desde la carpeta Notebooks del repositorio:

df = pd.read_csv("../data/Data_Mundial_2026.csv")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Preparación de datos

In [ ]:
df_resultados = df.dropna(subset=["home_score", "away_score"]).copy()

df_locales = pd.DataFrame({
    "equipo": df_resultados["home_team"],
    "goles": df_resultados["home_score"],
    "goles_rival": df_resultados["away_score"],
    "tiros_a_puerta": df_resultados["home_sot"],
    "posesion": df_resultados["home_possession"],
    "tiros_totales": df_resultados["home_total_shots"],
    "condicion": "Local"
})

df_visitantes = pd.DataFrame({
    "equipo": df_resultados["away_team"],
    "goles": df_resultados["away_score"],
    "goles_rival": df_resultados["home_score"],
    "tiros_a_puerta": df_resultados["away_sot"],
    "posesion": df_resultados["away_possession"],
    "tiros_totales": df_resultados["away_total_shots"],
    "condicion": "Visitante"
})

df_equipos = pd.concat([df_locales, df_visitantes], ignore_index=True)
df_equipos["resultado"] = np.select(
    [
        df_equipos["goles"] > df_equipos["goles_rival"],
        df_equipos["goles"] == df_equipos["goles_rival"],
        df_equipos["goles"] < df_equipos["goles_rival"]
    ],
    ["Victoria", "Empate", "Derrota"],
    default="Sin resultado"
)

orden_resultados = ["Victoria", "Empate", "Derrota"]
print(f"Partidos con marcador: {len(df_resultados)}")
print(f"Observaciones equipo-partido: {len(df_equipos)}")

## 3. Tiros a puerta según el resultado

Compara la mediana y la dispersión de tiros a puerta en victorias, empates y derrotas.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_equipos, x="resultado", y="tiros_a_puerta",
            order=orden_resultados, hue="resultado", legend=False)
plt.title("Tiros a puerta según el resultado del partido")
plt.xlabel("Resultado del equipo")
plt.ylabel("Tiros a puerta")
plt.tight_layout()
plt.show()

## 4. Posesión según el resultado

Permite explorar si la posesión presenta diferencias entre victorias, empates y derrotas. La posesión por sí sola no mide la calidad de las ocasiones.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_equipos, x="resultado", y="posesion",
            order=orden_resultados, hue="resultado", legend=False)
plt.title("Posesión según el resultado del partido")
plt.xlabel("Resultado del equipo")
plt.ylabel("Posesión del balón (%)")
plt.tight_layout()
plt.show()

## 5. Tiros a puerta y goles

La línea roja resume la tendencia lineal entre los tiros a puerta y los goles. Asociación no equivale a causalidad.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_equipos, x="tiros_a_puerta", y="goles",
            scatter_kws={"alpha": 0.65}, line_kws={"color": "red"})
plt.title("Relación entre tiros a puerta y goles marcados")
plt.xlabel("Tiros a puerta")
plt.ylabel("Goles marcados")
plt.tight_layout()
plt.show()

## 6. Tiros totales y goles

Compara el volumen total de disparos con los goles marcados.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_equipos, x="tiros_totales", y="goles",
            scatter_kws={"alpha": 0.65}, line_kws={"color": "red"})
plt.title("Relación entre tiros totales y goles marcados")
plt.xlabel("Tiros totales")
plt.ylabel("Goles marcados")
plt.tight_layout()
plt.show()

## 7. Eficacia de los tiros a puerta según el resultado

Indicador calculado como goles / tiros a puerta × 100. Si un equipo no registra tiros a puerta, la eficacia se deja como valor ausente porque la división no está definida. Con pocos tiros, un gol puede alterar mucho el porcentaje.

In [ ]:
df_equipos["eficacia_tiros"] = np.where(
    df_equipos["tiros_a_puerta"] > 0,
    df_equipos["goles"] / df_equipos["tiros_a_puerta"] * 100,
    np.nan
)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_equipos, x="resultado", y="eficacia_tiros",
            order=orden_resultados, hue="resultado", legend=False)
plt.title("Eficacia de los tiros a puerta según el resultado")
plt.xlabel("Resultado del equipo")
plt.ylabel("Goles por cada 100 tiros a puerta")
plt.tight_layout()
plt.show()

## 8. Porcentaje de victorias según la formación táctica

Se combinan las formaciones locales y visitantes. Solo se muestran formaciones con al menos tres observaciones; el umbral reduce casos muy pequeños, pero no garantiza representatividad.

In [ ]:
df_formaciones = pd.concat([
    pd.DataFrame({"equipo": df_resultados["home_team"],
                  "formacion": df_resultados["home_formation"],
                  "goles": df_resultados["home_score"],
                  "goles_rival": df_resultados["away_score"]}),
    pd.DataFrame({"equipo": df_resultados["away_team"],
                  "formacion": df_resultados["away_formation"],
                  "goles": df_resultados["away_score"],
                  "goles_rival": df_resultados["home_score"]})
], ignore_index=True)

df_formaciones["resultado"] = np.select(
    [
        df_formaciones["goles"] > df_formaciones["goles_rival"],
        df_formaciones["goles"] == df_formaciones["goles_rival"],
        df_formaciones["goles"] < df_formaciones["goles_rival"]
    ],
    ["Victoria", "Empate", "Derrota"],
    default="Sin resultado"
)

resumen_formaciones = (
    df_formaciones.dropna(subset=["formacion"])
    .groupby("formacion")
    .agg(
        partidos=("resultado", "size"),
        victorias=("resultado", lambda x: (x == "Victoria").sum())
    )
)
resumen_formaciones["porcentaje_victorias"] = (
    resumen_formaciones["victorias"] / resumen_formaciones["partidos"] * 100
)
resumen_formaciones = resumen_formaciones[
    resumen_formaciones["partidos"] >= 3
].sort_values("porcentaje_victorias", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=resumen_formaciones.reset_index(),
            x="formacion", y="porcentaje_victorias",
            hue="formacion", legend=False)
plt.title("Porcentaje de victorias según la formación táctica")
plt.xlabel("Formación táctica")
plt.ylabel("Victorias (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(resumen_formaciones.round(2))